<a href="https://colab.research.google.com/github/helenamrch/OAS5900-Masteroppgave/blob/main/name_context_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Name-in-Context Search

Searches for participant names from `'PLACEHOLDER'` in the `.md` protocol files
of each municipality subfolder. Returns paragraphs where names appear in context
(more than 3 words), filtering out standalone name-only lines.

**Output:** `'PLACEHOLDER'` with columns: `municipality, name, matched_text, filename`

In [ ]:
# ── Cell 1 ─ Imports & configuration ────────────────────────────────────────
import re
import unicodedata
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"'PLACEHOLDER'")
INPUT_XLSX = BASE_DIR / "'PLACEHOLDER'"
OUTPUT_CSV = BASE_DIR / "'PLACEHOLDER'"

print(f"Base dir : {BASE_DIR}")
print(f"Input    : {INPUT_XLSX}")
print(f"Output   : {OUTPUT_CSV}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Cell 2 ─ Load input & deduplicate ────────────────────────────────────────

raw = pd.read_excel(INPUT_XLSX, engine='openpyxl')
print(f"Raw rows: {len(raw)}")

# Keep only municipality + name, deduplicate
pairs = raw[['municipality', 'name']].dropna().drop_duplicates()
pairs = pairs.reset_index(drop=True)

# Normalize Unicode (NFC) so folder names match
pairs['municipality'] = pairs['municipality'].apply(
    lambda s: unicodedata.normalize('NFC', s)
)
pairs['name'] = pairs['name'].apply(
    lambda s: unicodedata.normalize('NFC', s)
)

print(f"Unique (municipality, name) pairs: {len(pairs)}")
print(f"Municipalities: {pairs['municipality'].nunique()}")
pairs.head(10)

In [ ]:
# ── Cell 3 ─ Search names in .md files ────────────────────────────────────────

def read_file(path):
    """Read a file, trying UTF-8 first then Latin-1."""
    try:
        return path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        return path.read_text(encoding='latin-1')


def extract_paragraphs(text):
    """
    Split text into paragraphs (separated by one or more blank lines).
    Strip markdown heading markers but keep heading text.
    """
    raw_paras = re.split(r'\n\s*\n', text)
    result = []
    for p in raw_paras:
        # Strip markdown headings
        cleaned = re.sub(r'^#+\s*', '', p.strip(), flags=re.MULTILINE)
        # Collapse internal whitespace
        cleaned = ' '.join(cleaned.split())
        if cleaned:
            result.append(cleaned)
    return result


# Group names by municipality for efficient processing
grouped = pairs.groupby('municipality')['name'].apply(list).to_dict()

results = []
municipalities_processed = 0
municipalities_missing = []

for muni, names in sorted(grouped.items()):
    muni_dir = BASE_DIR / muni
    if not muni_dir.is_dir():
        municipalities_missing.append(muni)
        continue

    md_files = sorted(muni_dir.glob('*.md'))
    if not md_files:
        continue

    # Pre-compile regex patterns for each name: full name + first name
    name_patterns = []
    for name in names:
        full_pat = re.compile(r'\b' + re.escape(name) + r'\b', re.IGNORECASE)
        first_name = name.split()[0]
        first_pat = re.compile(r'\b' + re.escape(first_name) + r'\b', re.IGNORECASE)
        name_patterns.append((name, full_pat, first_pat))

    for md_file in md_files:
        text = read_file(md_file)
        # Normalize Unicode in file content
        text = unicodedata.normalize('NFC', text)
        paragraphs = extract_paragraphs(text)

        for para in paragraphs:
            # Skip short paragraphs (3 words or fewer)
            if len(para.split()) <= 3:
                continue

            for name, full_pat, first_pat in name_patterns:
                if full_pat.search(para) or first_pat.search(para):
                    results.append({
                        'municipality': muni,
                        'name': name,
                        'matched_text': para,
                        'filename': md_file.name,
                    })

    municipalities_processed += 1
    if municipalities_processed % 10 == 0:
        print(f"  Processed {municipalities_processed} municipalities, {len(results)} matches so far...")

print(f"\nDone: {municipalities_processed} municipalities, {len(results)} total matches")
if municipalities_missing:
    print(f"WARNING – folders not found: {municipalities_missing}")

In [ ]:
# ── Cell 4 ─ Save CSV ─────────────────────────────────────────────────────────

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"Saved: {OUTPUT_CSV}")
print(f"Total matches    : {len(df)}")
print(f"Unique names     : {df['name'].nunique()}")
print(f"Municipalities   : {df['municipality'].nunique()}")
print(f"Files with hits  : {df['filename'].nunique()}")
print()
print("Matches per municipality:")
print(df.groupby('municipality').size().sort_values(ascending=False).to_string())
print()
df.head(20)

NameError: name 'results' is not defined